In [5]:
# %% [code] Cell 1: Dual-Arm Cooperative RL Environment (Optimized)
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import pybullet as p
import pybullet_utils.bullet_client as bullet_client

# --- OPTIMIZED MATH HELPERS ---
def get_quaternion_from_euler(euler_angles):
    """Fast C++ conversion for target orientations"""
    return p.getQuaternionFromEuler(euler_angles)

def get_angular_diff(q1, q2):
    """Vectorized angular difference using dot product"""
    q1, q2 = np.array(q1), np.array(q2)
    dot_prod = np.clip(np.abs(np.dot(q1, q2)), 0, 1)
    return 2 * np.arccos(dot_prod)

In [6]:
# --- DUAL-ARM ENVIRONMENT ---
class DualArmRobot(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "simulation_fps": 50}

    def __init__(self, render_mode="rgb_array"):
        super(DualArmRobot, self).__init__()
        self.render_mode = render_mode
        
        # 20 actions for two arms
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(20,), dtype=np.float32)
        # 45 observations: 2 dist, 2 ang_diff, 3 pos_t, 4 quat_t, 14 EE states, 20 joint angles
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(45,), dtype=np.float32)

        self.pos_tol = 0.15
        self.orient_tol = 1.0
        self.max_step_size = 250

        conn_mode = p.DIRECT if render_mode != "human" else p.GUI
        self._bullet_client = bullet_client.BulletClient(connection_mode=conn_mode)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_number = 0
        self._bullet_client.resetSimulation()
        self._bullet_client.setGravity(0, 0, -9.8)

        # Load the Dual-Arm URDF
        self.robot_id = self._bullet_client.loadURDF("TwoArms_col.urdf", [0, 0, 0], useFixedBase=True)

        # Generate a SINGLE shared Maternal Target
        self.pos_target = np.array([
            np.random.uniform(-0.05, 0.10), # Depth
            np.random.uniform(0.25, 0.65),  # 40cm Sweep
            np.random.uniform(0.10, 0.35)   # Height
        ], dtype=np.float32)

        euler_target = np.random.uniform([-2.35, -0.78, -np.pi], [-0.78, 0.78, np.pi])
        self.quat_target = get_quaternion_from_euler(euler_target)

        obs = self._get_obs()
        self.prev_dist = min(obs[0], obs[1]) # Start baseline with closest arm
        
        return obs, {}

    def _get_obs(self):
        # Link 10 (Left EE) and Link 20 (Right EE)
        state_L = self._bullet_client.getLinkState(self.robot_id, 10)
        state_R = self._bullet_client.getLinkState(self.robot_id, 20)
        
        pos_L, quat_L = np.array(state_L[0]), np.array(state_L[1])
        pos_R, quat_R = np.array(state_R[0]), np.array(state_R[1])

        dist_L = np.linalg.norm(pos_L - self.pos_target)
        dist_R = np.linalg.norm(pos_R - self.pos_target)

        q_diff_L = get_angular_diff(quat_L, self.quat_target)
        q_diff_R = get_angular_diff(quat_R, self.quat_target)

        # 20 Joint states
        joints = [self._bullet_client.getJointState(self.robot_id, i)[0] for i in range(20)]

        obs = np.concatenate([
            [dist_L, dist_R],                 # 2
            [q_diff_L, q_diff_R],             # 2
            self.pos_target,                  # 3
            self.quat_target,                 # 4
            pos_L, quat_L,                    # 7
            pos_R, quat_R,                    # 7
            joints                            # 20
        ]).astype(np.float32)                 # Total: 45
        return obs

    def check_collisions(self):
        contact_points = self._bullet_client.getContactPoints(bodyA=self.robot_id)
        return len(contact_points)

    def step(self, action):
        self.step_number += 1

        scaled_action = action * np.pi
        scaled_action[0] = action[0] / 4.0
        scaled_action[10] = action[10] / 4.0
        
        # Dependent joints mapping
        scaled_action[3] = -scaled_action[2]
        scaled_action[13] = -scaled_action[12]

        self._bullet_client.setJointMotorControlArray(
            self.robot_id, list(range(20)), p.POSITION_CONTROL, targetPositions=scaled_action
        )
        self._bullet_client.stepSimulation()

        obs = self._get_obs()
        dist_L, dist_R = obs[0], obs[1]
        q_diff_L, q_diff_R = obs[2], obs[3]

        # REWARD: Drive the closest arm toward the target
        current_min_dist = min(dist_L, dist_R)
        active_q_diff = q_diff_L if dist_L < dist_R else q_diff_R

        progress_reward = (self.prev_dist - current_min_dist) * 100.0
        reward = progress_reward - 0.02 - (active_q_diff * 0.05)
        self.prev_dist = current_min_dist

        # SOFT COLLISION PENALTY
        collisions = self.check_collisions()
        reward -= (collisions * 0.5)

        # SUCCESS LOGIC
        terminated = (current_min_dist < self.pos_tol) and (active_q_diff < self.orient_tol)
        if terminated:
            reward += 150.0

        truncated = self.step_number >= self.max_step_size
        
        info = {
            "is_success": terminated, 
            "min_dist": current_min_dist, 
            "collisions": collisions,
            "active_arm": "Left" if dist_L < dist_R else "Right"
        }

        return obs, reward, terminated, truncated, info


In [ ]:
# %% [code] Cell 2: Dual-Arm Cooperative SAC Training (With Logging)
import os
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

# 1. Initialize the Environment WITH the Monitor Wrapper
# We explicitly tell the Monitor to track our custom 'is_success' info key
def make_env():
    env = DualArmRobot(render_mode="direct")
    # Monitor logs episode rewards, lengths, and custom info keys
    return Monitor(env, info_keywords=("is_success",))

env = DummyVecEnv([make_env])

# 2. Normalize Observations (Crucial for 45 variables)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# 3. Define the "Thick" Neural Network Architecture for 20-DOF
policy_kwargs = dict(
    activation_fn=torch.nn.ReLU,
    net_arch=[256, 256, 256] 
)

# 4. Initialize the Brain
print("🧠 Initializing the 20-DOF Dual-Arm Master Brain...")
model = SAC(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=0.0003, 
    batch_size=256,
    ent_coef='auto',      
    verbose=1,
    device="auto"         
)

# 5. Start Training
print("🚀 Commencing Phase 2: Dual-Arm Cooperative Training (100,000 steps)...")
# Note: Bumped to 100k because 20-DOF needs more time to find the target
model.learn(total_timesteps=100000, progress_bar=True)

model.save("DualArm_Curriculum_Phase1")
env.save("vec_normalize_stats_phase1.pkl")

print("✅ Run Complete! Model and Stats Saved.")

🧠 Initializing the 20-DOF Dual-Arm Master Brain...

Using cpu device

🚀 Commencing Phase 2: Dual-Arm Cooperative Training (100,000 steps)...

argv[0]=


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 250      |
|    ep_rew_mean     | -55.4    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 95       |
|    time_elapsed    | 10       |
|    total_timesteps | 1000     |
| train/             |          |
|    actor_loss      | -53.5    |
|    critic_loss     | 0.647    |
|    ent_coef        | 0.764    |
|    ent_coef_loss   | -8.9     |
|    learning_rate   | 0.0003   |
|    n_updates       | 899      |
---------------------------------

...

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 204      |
|    ep_rew_mean     | -15.1    |
|    success_rate    | 0.19     |
| time/              |          |
|    episodes        | 260      |
|    fps             | 83       |
|    time_elapsed    | 638      |
|    total_timesteps | 53469    |
| train/             |          |
|    actor_loss      | -236     |
|    critic_loss     | 2.74     |
|    ent_coef        | 0.0303   |
|    ent_coef_loss   | -1.81    |
|    learning_rate   | 0.0003   |
|    n_updates       | 53368    |
---------------------------------

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 205      |
|    ep_rew_mean     | -18      |
|    success_rate    | 0.18     |
| time/              |          |
|    episodes        | 264      |
|    fps             | 83       |
|    time_elapsed    | 650      |
|    total_timesteps | 54469    |
| train/             |          |
|    actor_loss      | -233     |
|    critic_loss     | 3.21     |
|    ent_coef        | 0.0297   |
|    ent_coef_loss   | 0.225    |
|    learning_rate   | 0.0003   |
|    n_updates       | 54368    |
---------------------------------

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 203      |
|    ep_rew_mean     | -15.8    |
|    success_rate    | 0.19     |
| time/              |          |
|    episodes        | 268      |
|    fps             | 83       |
|    time_elapsed    | 660      |
|    total_timesteps | 55220    |
| train/             |          |
|    actor_loss      | -228     |
|    critic_loss     | 3.03     |
|    ent_coef        | 0.0291   |
|    ent_coef_loss   | 2.22     |
|    learning_rate   | 0.0003   |
|    n_updates       | 55119    |
---------------------------------

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 205      |
|    ep_rew_mean     | -18      |
|    success_rate    | 0.18     |
| time/              |          |
|    episodes        | 272      |
|    fps             | 83       |
|    time_elapsed    | 672      |
|    total_timesteps | 56220    |
| train/             |          |
|    actor_loss      | -225     |
|    critic_loss     | 2.8      |
|    ent_coef        | 0.0278   |
|    ent_coef_loss   | -0.664   |
|    learning_rate   | 0.0003   |
|    n_updates       | 56119    |
---------------------------------

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 208      |
|    ep_rew_mean     | -19.6    |
|    success_rate    | 0.17     |
| time/              |          |
|    episodes        | 276      |
|    fps             | 83       |
|    time_elapsed    | 684      |
|    total_timesteps | 57220    |
| train/             |          |
|    actor_loss      | -222     |
|    critic_loss     | 2.53     |
|    ent_coef        | 0.0276   |
|    ent_coef_loss   | -4.28    |
|    learning_rate   | 0.0003   |
|    n_updates       | 57119    |
---------------------------------